## Titanic Dataset (train.csv) — Pandas Only

This notebook follows the required workflow:

- **Step 1: Exploration**: `.head()`, `.info()`, `.describe()`
- **Step 2: Data Cleaning**:
  - `Age` → fill missing with **median**
  - `Embarked` → fill missing with **mode**
  - `Cabin` → **drop**
  - Remove duplicates (if any)
- **Step 3: Data Analysis (groupby)**:
  - Survival rate by gender
  - Survival rate by class
  - Average age per class
  - Survival rate by age group
- **Step 4: Filtering**:
  - Female passengers who survived
  - Children who survived
  - Passengers in 1st class with high survival probability
- **Step 5: Insights**: clear answers to the required questions

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

# Notebook is located in the `titanic ` folder, so we go one level up to `data/`
DATA_PATH = "../data/train.csv"

df = pd.read_csv(DATA_PATH)


## Step 1 — Exploration

In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [4]:
df.describe(include="all")

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
count,891.000000,891.000000,891.000000,891,891,714.000000,891.000000,891.000000,891,891.000000,204,889
unique,NaN,NaN,NaN,891,2,NaN,NaN,NaN,681,NaN,147,3
top,NaN,NaN,NaN,"Braund, Mr. Owen Harris",male,NaN,NaN,NaN,347082,NaN,G6,S
freq,NaN,NaN,NaN,1,577,NaN,NaN,NaN,7,NaN,4,644
mean,446.000000,0.383838,2.308642,NaN,NaN,29.699118,0.523008,0.381594,NaN,32.204208,NaN,NaN
std,257.353842,0.486592,0.836071,NaN,NaN,14.526497,1.102743,0.806057,NaN,49.693429,NaN,NaN
min,1.000000,0.000000,1.000000,NaN,NaN,0.420000,0.000000,0.000000,NaN,0.000000,NaN,NaN
25%,223.500000,0.000000,2.000000,NaN,NaN,20.125000,0.000000,0.000000,NaN,7.910400,NaN,NaN
50%,446.000000,0.000000,3.000000,NaN,NaN,28.000000,0.000000,0.000000,NaN,14.454200,NaN,NaN
75%,668.500000,1.000000,3.000000,NaN,NaN,38.000000,1.000000,0.000000,NaN,31.000000,NaN,NaN


## Step 2 — Data Cleaning

In [ ]:
clean_df = df.copy()

# Age: fill missing values with median age
age_median = clean_df["Age"].median()
clean_df["Age"].fillna(age_median, inplace=True)

# Embarked: fill missing values with mode (most frequent value)
embarked_mode = clean_df["Embarked"].mode(dropna=True)[0]
clean_df["Embarked"].fillna(embarked_mode, inplace=True)

# Cabin: drop this column entirely
if "Cabin" in clean_df.columns:
    clean_df.drop(columns=["Cabin"], inplace=True)

before_dedup = len(clean_df)
clean_df.drop_duplicates(inplace=True)
after_dedup = len(clean_df)

before_dedup, after_dedup

/tmp/ipykernel_22764/544205613.py:6: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  clean_df["Age"].fillna(age_median, inplace=True)
/tmp/ipykernel_22764/544205613.py:10: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works t

(891, 891)

## Step 3 — Data Analysis (groupby)

In [6]:
# Survival rate by gender
survival_by_gender = clean_df.groupby("Sex")["Survived"].mean()

# Survival rate by class
survival_by_class = clean_df.groupby("Pclass")["Survived"].mean()

# Average age per class
avg_age_per_class = clean_df.groupby("Pclass")["Age"].mean()

# Define age groups
age_bins = [0, 12, 18, 35, 50, 80]
age_labels = ["Child (0-12)", "Teen (13-18)", "Young Adult (19-35)", "Adult (36-50)", "Older (51+)"]
clean_df["AgeGroup"] = pd.cut(clean_df["Age"], bins=age_bins, labels=age_labels, right=True, include_lowest=True)

# Survival rate by age group
survival_by_age_group = clean_df.groupby("AgeGroup")["Survived"].mean()

survival_by_gender, survival_by_class, avg_age_per_class, survival_by_age_group

(Sex
 female    0.742038
 male      0.188908
 Name: Survived, dtype: float64,
 Pclass
 1    0.629630
 2    0.472826
 3    0.242363
 Name: Survived, dtype: float64,
 Pclass
 1    38.233441
 2    29.877630
 3    25.140620
 Name: Age, dtype: float64,
 AgeGroup
 Child (0-12)           0.579710
 Teen (13-18)           0.428571
 Young Adult (19-35)    0.382682
 Adult (36-50)          0.398693
 Older (51+)            0.343750
 Name: Survived, dtype: float64)

## Step 4 — Filtering

In [7]:
# Female passengers who survived
female_survivors = clean_df[(clean_df["Sex"] == "female") & (clean_df["Survived"] == 1)]

# Children who survived (Age < 12)
child_survivors = clean_df[(clean_df["Age"] < 12) & (clean_df["Survived"] == 1)]

# Passengers in 1st class with high survival probability
# Here we filter the cleaned data to 1st class and survived
first_class_high_survival = clean_df[(clean_df["Pclass"] == 1) & (clean_df["Survived"] == 1)]

female_survivors.head(), child_survivors.head(), first_class_high_survival.head()

(   PassengerId  Survived  Pclass  \
 1            2         1       1   
 2            3         1       3   
 3            4         1       1   
 8            9         1       3   
 9           10         1       2   
 
                                                 Name     Sex   Age  SibSp  \
 1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
 2                             Heikkinen, Miss. Laina  female  26.0      0   
 3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
 8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)  female  27.0      0   
 9                Nasser, Mrs. Nicholas (Adele Achem)  female  14.0      1   
 
    Parch            Ticket     Fare Embarked             AgeGroup  
 1      0          PC 17599  71.2833        C        Adult (36-50)  
 2      0  STON/O2. 3101282   7.9250        S  Young Adult (19-35)  
 3      0            113803  53.1000        S  Young Adult (19-35)  
 8      2            347742  1

## Step 5 — Insights

Below we summarize the key findings from the cleaned Titanic training data.

1. **Who was more likely to survive? (gender)**  
   We compare the survival rates for male vs female passengers using `survival_by_gender`.

2. **Did class affect survival?**  
   We inspect `survival_by_class` to see how survival changes across `Pclass` (1st, 2nd, 3rd).

3. **Were children prioritized?**  
   We look at survival rates by `AgeGroup`, focusing especially on the `Child (0-12)` group.

4. **What combination had the highest survival rate?**  
   We examine combinations of gender, class, and age group (e.g., female passengers in 1st class) to identify which group shows the highest survival.


In [8]:
# Compute concise numeric answers for the insights section

# 1. Survival rate by gender
survival_by_gender = clean_df.groupby("Sex")["Survived"].mean()

# 2. Survival rate by class
survival_by_class = clean_df.groupby("Pclass")["Survived"].mean()

# 3. Survival rate by age group (already computed, recompute to be explicit)
age_bins = [0, 12, 18, 35, 50, 80]
age_labels = ["Child (0-12)", "Teen (13-18)", "Young Adult (19-35)", "Adult (36-50)", "Older (51+)"]
clean_df["AgeGroup"] = pd.cut(clean_df["Age"], bins=age_bins, labels=age_labels, right=True, include_lowest=True)

survival_by_age_group = clean_df.groupby("AgeGroup")["Survived"].mean()

# 4. Highest survival combination (gender x class x age group)
combo_rates = (
    clean_df
    .groupby(["Sex", "Pclass", "AgeGroup"], dropna=True)["Survived"]
    .mean()
    .sort_values(ascending=False)
)

# Show the key aggregates to read in the markdown answers
survival_by_gender, survival_by_class, survival_by_age_group, combo_rates.head(10)

(Sex
 female    0.742038
 male      0.188908
 Name: Survived, dtype: float64,
 Pclass
 1    0.629630
 2    0.472826
 3    0.242363
 Name: Survived, dtype: float64,
 AgeGroup
 Child (0-12)           0.579710
 Teen (13-18)           0.428571
 Young Adult (19-35)    0.382682
 Adult (36-50)          0.398693
 Older (51+)            0.343750
 Name: Survived, dtype: float64,
 Sex     Pclass  AgeGroup           
 female  1       Teen (13-18)           1.000000
         2       Child (0-12)           1.000000
         1       Older (51+)            1.000000
         2       Teen (13-18)           1.000000
 male    1       Child (0-12)           1.000000
 female  3       Older (51+)            1.000000
 male    2       Child (0-12)           1.000000
 female  1       Young Adult (19-35)    0.972222
                 Adult (36-50)          0.960000
         2       Young Adult (19-35)    0.925000
 Name: Survived, dtype: float64)

In [9]:
# Step 5 — Turn the groupby results into clear answers

# 1) Who was more likely to survive? (gender)
more_likely_gender = survival_by_gender.idxmax()
more_likely_gender_rate = survival_by_gender.max()

# 2) Did class affect survival?
best_class = survival_by_class.idxmax()
best_class_rate = survival_by_class.max()
worst_class = survival_by_class.idxmin()
worst_class_rate = survival_by_class.min()

# 3) Were children prioritized? (compare children vs non-children average)
child_label = "Child (0-12)"
child_rate = survival_by_age_group.loc[child_label] if child_label in survival_by_age_group.index else float("nan")
non_child_mean = survival_by_age_group.drop(index=child_label).mean() if child_label in survival_by_age_group.index else float("nan")

# 4) What combination had the highest survival rate? (gender x class x age group)
(best_sex, best_pclass, best_agegroup) = combo_rates.index[0]
best_combo_rate = combo_rates.iloc[0]

# Print concise, readable answers
print("1) More likely to survive (gender):")
print(f"   {more_likely_gender} passengers had a survival rate of {more_likely_gender_rate:.2%}.")

print("\n2) Did class affect survival?")
print(f"   Best class: {int(best_class)} with {best_class_rate:.2%} survival.")
print(f"   Worst class: {int(worst_class)} with {worst_class_rate:.2%} survival.")

print("\n3) Were children prioritized?")
if child_label in survival_by_age_group.index:
    print(f"   Children ({child_label}) survival rate: {child_rate:.2%}.")
    print(f"   Non-children average survival rate: {non_child_mean:.2%}.")
    if child_rate > non_child_mean:
        print("   Conclusion: children had higher survival than non-children in this dataset.")
    elif child_rate < non_child_mean:
        print("   Conclusion: children had lower survival than non-children in this dataset.")
    else:
        print("   Conclusion: children and non-children had the same survival rate.")
else:
    print("   Age-group labels do not include the expected child bucket; cannot compare children vs non-children.")

print("\n4) Highest survival combination:")
print(f"   Sex={best_sex}, Pclass={int(best_pclass)}, AgeGroup={best_agegroup} => {best_combo_rate:.2%} survival.")


1) More likely to survive (gender):
   female passengers had a survival rate of 74.20%.

2) Did class affect survival?
   Best class: 1 with 62.96% survival.
   Worst class: 3 with 24.24% survival.

3) Were children prioritized?
   Children (Child (0-12)) survival rate: 57.97%.
   Non-children average survival rate: 38.84%.
   Conclusion: children had higher survival than non-children in this dataset.

4) Highest survival combination:
   Sex=female, Pclass=1, AgeGroup=Teen (13-18) => 100.00% survival.
